# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHITCRAFTSYT/flyrank-int/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. The contract, in plain words (answers 1–3)

My lane is **CTR / Engagement Opportunity Scoring** (set in ML-02/03). Here is the slice of the warehouse it uses, in plain words. The code cell below states nothing new — it *verifies* these three claims on real data.

**1 — What one row means.** One row = **one pseudonymized page** (`client_hash_id × content_hash_id`), summarised over a single observation month. The **decision moment is the end of that month**: everything in a row is something a content editor could know on 2026-03-31 when deciding "which page do I open first?". Not one row per day (an editor doesn't act on a Tuesday); not one row per client (you can't edit a client); not one row per query (the release ships no raw queries).

**2 — Which table(s).** The spine is **`fact_content_daily_performance`**, read at the **`month=2026-03` partition only** (grain: `report_date × client × content`). I aggregate its daily rows up to one row per page. **`dim_clients`** supplies per-client history (`gsc_data_start`) and the availability flags; **`dim_content`** is available for static attributes via `content_hash_id`. I do **not** use `fact_content_query_90d` here — its fixed 90-day window would overlap and complicate a clean single-month contract.

**3 — Which time window.** The **31 days of March 2026** are the observation/feature window; the decision moment is **2026-03-31**. March is a mid-panel month chosen on purpose: the `_sample` table (June 2026) is the natural outcome window of any past→future label, so it stays **sealed** and is never queried here. Because the panel is unbalanced, only clients whose `gsc_data_start ≤ 2026-03-01` have a genuinely full month — section 4 measures how many don't.

*(Answers 4 (predict/rank) and 5 (deliberate exclusion) are in section 2, where the field buckets make them precise.)*


In [ ]:
# ── Setup: connect DuckDB to the gated warehouse, point at ONE middle month ──
# Access pattern reused verbatim from notebooks/03_working_with_the_full_release.ipynb.
# Token via getpass / Colab Secret ONLY — never pasted in a cell (this repo is PUBLIC).
%pip -q install duckdb huggingface_hub
import os, getpass
import pandas as pd, numpy as np
pd.set_option("display.width", 160)

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL   = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"                       # mid-panel month. June 2026 (_sample) stays SEALED.
DAILY       = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

# Confirm we are pointed at the right thing (COUNT + MIN/MAX touch metadata — near-free).
head = con.sql(f"SELECT COUNT(*) AS daily_rows, MIN(report_date) AS lo, MAX(report_date) AS hi FROM {DAILY}").df()
print("=== month=2026-03 partition, sanity ===")
print(head.to_string(index=False), "\n")

# Print the REAL column names so nothing below is guessed (GA4 / flag names vary by build).
print("=== columns in fact_content_daily_performance ===")
print(con.sql(f"DESCRIBE SELECT * FROM {DAILY}").df()[["column_name", "column_type"]].to_string(index=False))


## 2. Fields: feature / label / context / excluded (answers 4–5)

Every column I touch goes in exactly one bucket. The code cell below **builds** the frame this describes.

**Feature — knowable BEFORE the decision moment (2026-03-31), safe to use.** Five, all aggregated from GSC daily rows inside March:

| Feature | Knowable at the decision moment because… |
|---|---|
| `imp_30d` | search impressions summed inside March — **exposure** measured in the window, not an outcome |
| `avg_position_30d` | impression-weighted rank inside March — **where the page already sits**; built from position, not clicks |
| `active_days` | March days with ≥1 impression — **coverage** measured in the window |
| `position_volatility` | std of daily rank inside March — a **within-window stability** measurement |
| `imp_trend_ratio` | March last-15d ÷ first-15d impressions — **both halves end on/before 2026-03-31** |

**Label / proxy (answer 4 — what I rank).** I rank by **`ctr_gap`** = `ctr_30d − median(ctr_30d) within position band` — how far a page's March click-through sits below comparable pages at the same rank (negative = under-capturing). This is **defined, not observed** — a legitimate *baseline*, exactly the proxy from ML-03. Its parents `ctr_30d`, `clicks`, `expected_ctr` are the label family and are **never features**. The genuinely-observed target (did clicks rise in a strictly-later window, position held) needs a forward month and is the optional sibling notebook's job — not this one.

**Context — join / group / split only, never learned from.** `client_hash_id`, `content_hash_id` (pseudonyms); `pos_band` (grouping key for the expected-CTR curve). `client_hash_id` is also my grouped-split key so no client sits in both train and test.

**Excluded (answer 5 — the one thing I deliberately leave out).** **Pages with zero clicks in March**, dropped by a `clicks ≥ 1` floor (plus `impressions ≥ 500` so CTR isn't noise). This is the contract decision I promised myself in ML-03: a page at position ~4 with thousands of impressions and *zero* clicks over a month is what broken click-tracking looks like, not a content problem, and it clustered by client. **The cost, stated honestly:** the floor also drops any genuinely-broken page that truly earned zero clicks, and it does not *fix* partial-tracking clients — a per-client tracking-sanity check is the deeper option I'm deferring to the capstone. Separately, the label family (`clicks`, `ctr_30d`, `ctr_gap`, `expected_ctr`) is excluded *as features* — section 3's trap shows what happens if I forget.


In [ ]:
# ── Build the unit of analysis: one row = one page, over month=2026-03 ──
# Heavy lifting in SQL (aggregate daily -> page); only the small frame comes back to pandas.
# FEATURES use GSC exposure / position / coverage only — NEVER gsc_clicks or ctr (label's parents).
frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        -- feature inputs (all knowable within the observation month) --
        SUM(gsc_impressions)                                                 AS imp_30d,
        SUM(gsc_impressions * gsc_avg_position)
            / NULLIF(SUM(gsc_impressions), 0)                               AS avg_position_30d,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)       AS active_days,
        STDDEV_SAMP(gsc_avg_position) FILTER (WHERE gsc_impressions > 0)     AS position_volatility,
        SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-16') AS imp_last15,
        SUM(gsc_impressions) FILTER (WHERE report_date <  DATE '2026-03-16') AS imp_first15,
        -- label ingredient (NOT a feature) --
        SUM(gsc_clicks)                                                      AS clk_30d
    FROM {DAILY}
    GROUP BY 1, 2
    -- deliberate scope exclusion (contract answer 5): >=500 imp so CTR isn't noise,
    -- >=1 click to drop zero-click pages (ML-03 finding: broken tracking, not content).
    HAVING SUM(gsc_impressions) >= 500 AND SUM(gsc_clicks) >= 1
""").df()

# Visible pages only (mirror ML-02 universe): a real rank where a click is possible.
frame = frame[(frame.avg_position_30d > 0) & (frame.avg_position_30d <= 20)].copy()

# Fifth feature: within-month momentum (both halves lie inside the observation window).
frame["imp_trend_ratio"] = frame.imp_last15 / frame.imp_first15.replace(0, np.nan)

# ── The proxy LABEL: position-adjusted CTR deficit (defined, not observed) ──
frame["ctr_30d"]      = 100 * frame.clk_30d / frame.imp_30d          # percent units, like the starter
frame["pos_band"]     = frame.avg_position_30d.round().clip(1, 20).astype(int)
frame["expected_ctr"] = frame.groupby("pos_band")["ctr_30d"].transform("median")
frame["ctr_gap"]      = frame.ctr_30d - frame.expected_ctr          # negative = under-capturing

FEATURES = ["imp_30d", "avg_position_30d", "active_days", "position_volatility", "imp_trend_ratio"]

print(f"pages in my March slice: {len(frame):,}  across {frame.client_hash_id.nunique()} clients\n")
print("=== the five features, and why each is knowable AT the decision moment (2026-03-31) ===")
_why = {
    "imp_30d":             "impressions summed inside March — exposure in the window, not an outcome",
    "avg_position_30d":    "impression-weighted March rank — where the page already sits; from position, not clicks",
    "active_days":         "count of March days with >=1 impression — coverage measured in the window",
    "position_volatility": "std of daily rank inside March — a within-window stability measurement",
    "imp_trend_ratio":     "March last-15d / first-15d impressions — both halves end on/before 2026-03-31",
}
for f in FEATURES:
    print(f"  - {f:<20} knowable because {_why[f]}")
print("\nEXCLUDED from features (label family, would leak): gsc_clicks, ctr_30d, expected_ctr, ctr_gap")
frame[["client_hash_id", "content_hash_id", *FEATURES, "ctr_30d", "ctr_gap"]].head()


## 3. Verify it with queries — grain, count/span, availability — then spring the trap

A contract claim without a query next to it is a guess. Three checks, each backing a sentence above, then the deliberate leak.

- **Q1 · Grain.** `GROUP BY report_date, client, content HAVING COUNT(*) > 1` must return **0 rows** — one daily row really is one date×client×content. Then I re-check my aggregated frame is one row per page.
- **Q2 · Count + date span.** Raw daily rows in the partition, my slice's page count, and `MIN/MAX(report_date)` — which must land inside **2026-03-01 … 2026-03-31**.
- **Q3 · Availability, with `IS TRUE`.** `ga4_data_available` is not just TRUE/FALSE — millions of rows are **NULL**, and NULL is neither. I count `IS TRUE` / `= FALSE` / `IS NULL` separately and show they sum to the total, so `NOT flag` / `= FALSE` would silently miscount. (GA4 gates any engagement feature; my five features are GSC-based, but this is the flag the data skill explicitly warns about.)

**The trap (the leakage lesson from notebook 02, on real March data).** I build a quick client-grouped score for "deserves review" (worst-quartile `ctr_gap` within band). First with my **five honest features** → a modest number. Then I add **one** label-derived column, `ctr_30d` → the score jumps toward perfect, because the label is built from `ctr`. Then I **delete it and keep the honest number**. Nothing is learned by the leak except how to read the answer key.


In [ ]:
# ── Section 3: verify every contract claim with a query, then spring the trap ──
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

# Q1 — GRAIN: is one daily row really report_date × client × content? (expect 0 rows)
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {DAILY}
    GROUP BY 1, 2, 3 HAVING COUNT(*) > 1 LIMIT 5
""").df()
print("=== Q1 grain probe ===")
print(f"  daily rows with a duplicate (date,client,content) key : {len(q1)}   <- must be 0")
assert len(q1) == 0, "grain broken: (report_date, client, content) is not unique"
dupe_pages = int(frame.duplicated(["client_hash_id", "content_hash_id"]).sum())
print(f"  duplicate pages in my aggregated frame                 : {dupe_pages}   <- must be 0")
assert dupe_pages == 0
print("  -> grain confirmed: daily = date×client×content; my frame = one row per page.\n")

# Q2 — COUNT + DATE SPAN
q2 = con.sql(f"SELECT COUNT(*) AS daily_rows, MIN(report_date) AS lo, MAX(report_date) AS hi FROM {DAILY}").df()
print("=== Q2 row count + date span ===")
print(f"  raw daily rows in month partition        : {int(q2.daily_rows[0]):,}")
print(f"  date span                                : {q2.lo[0]} -> {q2.hi[0]}   (expect 2026-03-01..2026-03-31)")
print(f"  MY SLICE (visible, >=500 imp, >=1 click) : {len(frame):,} pages, {frame.client_hash_id.nunique()} clients\n")

# Q3 — AVAILABILITY with IS TRUE (NULL is neither TRUE nor FALSE)
q3 = con.sql(f"""
    SELECT COUNT(*)                                            AS total,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS is_true,
           COUNT(*) FILTER (WHERE ga4_data_available = FALSE)  AS eq_false,
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS is_null
    FROM {DAILY}
""").df()
t, tt, ff, nn = (int(q3[c][0]) for c in ["total", "is_true", "eq_false", "is_null"])
print("=== Q3 availability — filter with IS TRUE, count survivors ===")
print(f"  ga4_data_available IS TRUE : {tt:,}   <- rows that survive (real GA4 measurement)")
print(f"  ga4_data_available = FALSE : {ff:,}")
print(f"  ga4_data_available IS NULL : {nn:,}")
print(f"  check: {tt:,} + {ff:,} + {nn:,} = {tt+ff+nn:,}  == total {t:,}")
print(f"  the trap: `NOT (flag IS TRUE)` would wrongly grab {ff+nn:,} rows ({nn:,} of them NULL).")
print("  -> always write `ga4_data_available IS TRUE`; `= FALSE` / `NOT flag` silently miscount.\n")

# ── THE TRAP: add ONE label-derived column, watch the score jump, then delete it ──
# 'Deserves review' proxy label: worst-quartile ctr_gap WITHIN its position band (per-band,
# so it can never just mean 'ranks badly'). Notebook-02's lesson, on real March data.
band_q25 = frame.groupby("pos_band")["ctr_gap"].transform(lambda s: s.quantile(0.25))
frame["underperf"] = (frame.ctr_gap <= band_q25).astype(int)

def grouped_auc(cols):
    X = frame[cols].fillna(frame[cols].median(numeric_only=True))
    y, g = frame.underperf.values, frame.client_hash_id.values
    aucs = []
    for tr, te in GroupKFold(n_splits=5).split(X, y, g):
        if len(np.unique(y[te])) < 2:               # skip a fold with only one class
            continue
        m = RandomForestClassifier(n_estimators=120, min_samples_leaf=20, random_state=0, n_jobs=-1)
        m.fit(X.iloc[tr], y[tr])
        aucs.append(roc_auc_score(y[te], m.predict_proba(X.iloc[te])[:, 1]))
    return float(np.mean(aucs))

honest = grouped_auc(FEATURES)                       # 5 safe features, client-grouped
leaked = grouped_auc(FEATURES + ["ctr_30d"])         # + the label's parent
print("=== quick score (client-grouped ROC-AUC), with vs without the leak ===")
print(f"  HONEST  (5 safe features)        : {honest:.3f}")
print(f"  +ctr_30d (label-derived, LEAKED) : {leaked:.3f}   <- jumps toward perfect")
print(f"  the jump ({leaked - honest:+.3f}) is the tell: ctr_30d is what the label is built from.")
print(f"  -> DELETE the leak. The honest number I keep and report is {honest:.3f}, not {leaked:.3f}.")


## 4. Data limits — one named limitation

**My March slice cannot speak for every client — it is an unbalanced panel.** The daily history starts whenever each client's tracking started, so a client whose `gsc_data_start` falls *after* 2026-03-31 (or is unknown) contributes **zero rows** to my window. Any ranking I build is therefore silent about those clients — not because they're fine, but because they're absent. The code below counts exactly how many. This is a property of the data, not a modelling choice, and it compounds the ML-03 finding that even *present* clients differ in tracking quality (the zero-click clusters) — so "covered in my slice" is not the same as "measured well." Both go in the capstone's caveats, and both argue for **per-client windows and per-client checks** rather than one global calendar month.


In [ ]:
# ── Section 4: the named limitation, backed by a query ──
cov = con.sql(f"""
    SELECT COUNT(*)                                                       AS total_clients,
           COUNT(*) FILTER (WHERE gsc_data_start IS NULL
                               OR gsc_data_start > DATE '2026-03-31')     AS absent_in_march
    FROM {DIM_CLIENTS}
""").df()
tot, absent = int(cov.total_clients[0]), int(cov.absent_in_march[0])
present = frame.client_hash_id.nunique()
print("=== Limitation: my March slice cannot speak for every client ===")
print(f"  clients in dim_clients                       : {tot}")
print(f"  clients with GSC history starting AFTER March: {absent}   (0 rows in my window)")
print(f"  clients actually present in my slice         : {present}")
print("  -> the ranking is silent about absent clients (unbalanced panel), and per the")
print("     ML-03 zero-click finding, present clients still differ in tracking quality.")
print("     'Covered in my slice' != 'measured well'. Capstone: per-client windows + checks.")


## Self-check

> ⚠️ **You must run this yourself before submitting.** The cells were drafted with an AI assistant but not executed here — the warehouse is gated behind *your* `HF_TOKEN`. Open in Colab, add `HF_TOKEN` as a Secret, **Runtime → Run all**, read every output, then tick the last two boxes and commit. If a column name differs, the `DESCRIBE` output in section 1 is your source of truth — fix and re-run.

- [x] Every section is filled — plain-words contract (§1–2) AND the code that verifies it (§3–4)
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **you confirm this**
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `*_hash_id` and aggregates appear; the release ships no raw URL/title/query columns
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### The contract in one table

| Piece | My answer |
|---|---|
| **One row** | one page (`client_hash_id × content_hash_id`), summarised over month=2026-03; decision moment = 2026-03-31 |
| **Table(s)** | `fact_content_daily_performance` (month=2026-03) as spine; `dim_clients` for history/flags; `dim_content` available for attributes |
| **Window** | 31 days of March 2026; June 2026 (`_sample`) sealed as the future |
| **Rank by (label/proxy)** | `ctr_gap` = position-adjusted CTR deficit — **defined, not observed**; a baseline. Real forward label deferred |
| **Deliberately excluded** | zero-click pages (`clicks ≥ 1` floor, `imp ≥ 500`) — likely broken tracking (ML-03); and the label family as features |
| **Verified** | grain (0 dup keys), count + span (March only), availability (`IS TRUE`, NULLs counted apart) |
| **Named limitation** | unbalanced panel — clients starting after March contribute 0 rows; coverage ≠ data quality |

### What the trap taught, in one line

The five honest features earn a modest client-grouped AUC; adding `ctr_30d` — the column the label is built from — jumps it toward perfect while learning nothing but the answer key. On the warehouse this is the same lesson as notebook 02, now on data I aggregated myself. **I keep the honest number.**

### Carried forward to ML-05 / capstone

1. **Forward label with position held constant** — the observed target this proxy stands in for; needs a strictly-later window (the sibling `w03_feature_leakage_check.ipynb`).
2. **Per-client tracking-sanity check** — better than a blanket `clicks ≥ 1` floor; is the zero-click pattern broken tracking or real?
3. **Is the position band the right comparison group?** — `fact_content_query_90d` can test whether a page's "average position" hides a wide per-query spread.
